In [1]:
import pandas as pd
import numpy as np
import openmatrix as omx
import geopandas as gpd

In [2]:
data_path = "../data"
zone_seq = pd.read_csv(f"{data_path}/mtc_final_network_zone_seq_ver12.csv")
tap_seq = zone_seq[zone_seq["TAPSEQ"] > 0].drop(columns=["TAZSEQ", "MAZSEQ", "EXTSEQ"]).rename(columns={"N": "tap", "TAPSEQ": "renum_tap"}).reset_index(drop=True)
tap_seq.head()

,tap,renum_tap
0,90001,1
1,90002,2
2,90003,3
3,90004,4
4,90005,5


In [3]:
maz_nodes = gpd.read_file(f"{data_path}/maz_nodes.gpkg")
maz_nodes = maz_nodes.to_crs(4326)
maz_nodes["X"] = maz_nodes["geometry"].apply(lambda p: p.x)
maz_nodes["Y"] = maz_nodes["geometry"].apply(lambda p: p.y)
maz_nodes = maz_nodes[["N", "X", "Y"]].rename(columns={"N": "maz", "X": "lon", "Y": "lat"})
maz_nodes.head()

,maz,lon,lat
0,10001,-122.441076,37.750066
1,10002,-122.438934,37.750197
2,10003,-122.436270,37.756793
3,10004,-122.437049,37.753528
4,10005,-122.433959,37.755725


### Create skim db for TAP approach
#### import reformatted on-board survey record for the TAP approach

In [4]:
tap_base = pd.read_csv(f"{data_path}/tap_approach/tap_approach_base_table.csv")
tap_base.head()

,unique_ID,maz_o,maz_d,period,skim_set,tap_o,tap_d,access_time,egress_time
0,11420___AC Transit___2018,413852,414639,EA,1,490058,490309,0.057,0.190
1,11423___AC Transit___2018,424701,414651,EA,1,490040,490462,0.054,0.088
2,11425___AC Transit___2018,418818,424101,EA,1,490074,490016,0.092,0.021
3,11730___AC Transit___2018,322305,329001,EA,1,390570,390327,0.253,0.319
4,11732___AC Transit___2018,310894,329001,EA,1,390659,390605,0.052,0.042


In [5]:
# change access & egress time unit from hours to minutes
tap_base["access_time"] = tap_base["access_time"] * 60
tap_base["egress_time"] = tap_base["egress_time"] * 60
tap_base.head()

,unique_ID,maz_o,maz_d,period,skim_set,tap_o,tap_d,access_time,egress_time
0,11420___AC Transit___2018,413852,414639,EA,1,490058,490309,3.42,11.40
1,11423___AC Transit___2018,424701,414651,EA,1,490040,490462,3.24,5.28
2,11425___AC Transit___2018,418818,424101,EA,1,490074,490016,5.52,1.26
3,11730___AC Transit___2018,322305,329001,EA,1,390570,390327,15.18,19.14
4,11732___AC Transit___2018,310894,329001,EA,1,390659,390605,3.12,2.52


In [6]:
# add renumbered tap num
tap_base = pd.merge(tap_base, tap_seq, how="left", left_on="tap_o", right_on="tap").drop(columns=["tap"]).rename(columns={"renum_tap": "renum_tap_o"})
tap_base = pd.merge(tap_base, tap_seq, how="left", left_on="tap_d", right_on="tap").drop(columns=["tap"]).rename(columns={"renum_tap": "renum_tap_d"})
tap_base.head()

,unique_ID,maz_o,maz_d,period,skim_set,tap_o,tap_d,access_time,egress_time,renum_tap_o,renum_tap_d
0,11420___AC Transit___2018,413852,414639,EA,1,490058,490309,3.42,11.40,3946,4197
1,11423___AC Transit___2018,424701,414651,EA,1,490040,490462,3.24,5.28,3928,4350
2,11425___AC Transit___2018,418818,424101,EA,1,490074,490016,5.52,1.26,3962,3904
3,11730___AC Transit___2018,322305,329001,EA,1,390570,390327,15.18,19.14,3034,2791
4,11732___AC Transit___2018,310894,329001,EA,1,390659,390605,3.12,2.52,3123,3069


#### add corresponding tap-to-tap skim value from omx matrix

In [7]:
tap_skim_ea = omx.open_file(f"{data_path}/tap_approach/skims/transit_skims_ea.omx")
tap_skim_am = omx.open_file(f"{data_path}/tap_approach/skims/transit_skims_am.omx")
tap_skim_md = omx.open_file(f"{data_path}/tap_approach/skims/transit_skims_md.omx")
tap_skim_pm = omx.open_file(f"{data_path}/tap_approach/skims/transit_skims_pm.omx")
tap_skim_ev = omx.open_file(f"{data_path}/tap_approach/skims/transit_skims_ev.omx")

- table name starts with `<period>_BUS` is skim set 1
- table name starts with `<period>_PREM` is skim set 2
- table name starts with `<period>_ALLPEN` is skim set 3

In [8]:
def get_tap_skim_value(period, skim_set, tap_o, tap_d):
    # initialize result dictionary
    table_names = ["CAPPEN", "CRIVTT", "CROWD", "EAWT", "EBIVTT", "FARE", "FIRSTWAIT", "FRIVTT", "HRIVTT", "LBIVTT", "LINKREL", "LRIVTT", "TOTALIVTT", "TOTALWAIT", "TOTALWALK", "XFERS", "XFERWAIT", "XFERWALK",]
    result = {}

    # select which omx skim file to use
    if period == "EA":
        per_skim = tap_skim_ea
    elif period == "AM":
        per_skim = tap_skim_am
    elif period == "MD":
        per_skim = tap_skim_md
    elif period == "PM":
        per_skim = tap_skim_pm
    elif period == "EV":
        per_skim = tap_skim_ev
    
    # set string for filtering appropriate skim table based on skim_set argument
    if skim_set == 1:
        skim_set_str = "BUS"
    elif skim_set == 2:
        skim_set_str = "PREM"
    elif skim_set == 3:
        skim_set_str = "ALLPEN"

    # populate result dictionary
    for table in table_names:
        result[table] = per_skim[f"{period.lower()}_{skim_set_str}_{table}"][tap_o - 1, tap_d - 1] # -1 because matrix starts from index 0
    
    return result

In [9]:
def add_tap_to_tap_skim_values(base_table):

    skim_value_list = []
    for i in range(len(base_table)):
        period = base_table.loc[i, "period"]
        skim_set = base_table.loc[i, "skim_set"]
        tap_o = base_table.loc[i, "renum_tap_o"]
        tap_d = base_table.loc[i, "renum_tap_d"]
        
        skim_value_list.append(get_tap_skim_value(period, skim_set, tap_o, tap_d))

    skim_values = pd.DataFrame(skim_value_list)

    return pd.concat([base_table, skim_values], axis = 1)
    

In [10]:
tap_approach_db = add_tap_to_tap_skim_values(tap_base)
print(len(tap_approach_db))
tap_approach_db.head()

85209


,unique_ID,maz_o,maz_d,period,skim_set,tap_o,tap_d,access_time,egress_time,renum_tap_o,...,HRIVTT,LBIVTT,LINKREL,LRIVTT,TOTALIVTT,TOTALWAIT,TOTALWALK,XFERS,XFERWAIT,XFERWALK
0,11420___AC Transit___2018,413852,414639,EA,1,490058,490309,3.42,11.40,3946,...,0.0,0.000000,0.0,0.0,0.000000,45.0,16.571501,3.0,30.0,15.911501
1,11423___AC Transit___2018,424701,414651,EA,1,490040,490462,3.24,5.28,3928,...,0.0,11.970170,0.0,0.0,11.970170,30.0,10.805839,2.0,15.0,10.145839
2,11425___AC Transit___2018,418818,424101,EA,1,490074,490016,5.52,1.26,3962,...,0.0,10.733758,0.0,0.0,10.733758,30.0,2.304314,1.0,15.0,1.644314
3,11730___AC Transit___2018,322305,329001,EA,1,390570,390327,15.18,19.14,3034,...,0.0,18.925690,0.0,0.0,18.925690,15.0,0.400000,0.0,0.0,0.000000
4,11732___AC Transit___2018,310894,329001,EA,1,390659,390605,3.12,2.52,3123,...,0.0,2.697566,0.0,0.0,2.697566,15.0,0.400000,0.0,0.0,0.000000


In [11]:
# add total time column
tap_approach_db["TOTAL_TIME"] = tap_approach_db["access_time"] + tap_approach_db["egress_time"] + tap_approach_db["TOTALIVTT"] + tap_approach_db["TOTALWAIT"] + tap_approach_db["XFERWALK"]
tap_approach_db.head()

,unique_ID,maz_o,maz_d,period,skim_set,tap_o,tap_d,access_time,egress_time,renum_tap_o,...,LBIVTT,LINKREL,LRIVTT,TOTALIVTT,TOTALWAIT,TOTALWALK,XFERS,XFERWAIT,XFERWALK,TOTAL_TIME
0,11420___AC Transit___2018,413852,414639,EA,1,490058,490309,3.42,11.40,3946,...,0.000000,0.0,0.0,0.000000,45.0,16.571501,3.0,30.0,15.911501,75.731501
1,11423___AC Transit___2018,424701,414651,EA,1,490040,490462,3.24,5.28,3928,...,11.970170,0.0,0.0,11.970170,30.0,10.805839,2.0,15.0,10.145839,60.636009
2,11425___AC Transit___2018,418818,424101,EA,1,490074,490016,5.52,1.26,3962,...,10.733758,0.0,0.0,10.733758,30.0,2.304314,1.0,15.0,1.644314,49.158072
3,11730___AC Transit___2018,322305,329001,EA,1,390570,390327,15.18,19.14,3034,...,18.925690,0.0,0.0,18.925690,15.0,0.400000,0.0,0.0,0.000000,68.245690
4,11732___AC Transit___2018,310894,329001,EA,1,390659,390605,3.12,2.52,3123,...,2.697566,0.0,0.0,2.697566,15.0,0.400000,0.0,0.0,0.000000,23.337566


In [12]:
# if TOTALIVTT == 0, set TOTAL_TIME = "NA"
# for walk-only paths, set all skim values to null
for column_name in ["access_time", "egress_time", "CAPPEN", "CRIVTT", "CROWD", "EAWT", "EBIVTT", "FARE", "FIRSTWAIT", "FRIVTT", "HRIVTT", "LBIVTT", "LINKREL", "LRIVTT", "TOTALWAIT", "TOTALWALK", "XFERS", "XFERWAIT", "XFERWALK", "TOTAL_TIME"]:
    tap_approach_db.loc[tap_approach_db["TOTALIVTT"] == 0, column_name] = np.nan
temp = tap_approach_db[tap_approach_db["TOTALIVTT"] == 0]
temp.head()

,unique_ID,maz_o,maz_d,period,skim_set,tap_o,tap_d,access_time,egress_time,renum_tap_o,...,LBIVTT,LINKREL,LRIVTT,TOTALIVTT,TOTALWAIT,TOTALWALK,XFERS,XFERWAIT,XFERWALK,TOTAL_TIME
0,11420___AC Transit___2018,413852,414639,EA,1,490058,490309,NaN,NaN,3946,...,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN
66,3295___AC Transit___2018,310449,311366,EA,1,391021,390524,NaN,NaN,3485,...,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN
137,671___LAVTA___2018,312402,317158,EA,1,390242,390959,NaN,NaN,2706,...,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN
220,1326___SF Muni___2017,17323,13034,EA,1,90279,90151,NaN,NaN,279,...,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN
413,2012___SF Muni___2017,118507,14635,EA,1,190192,90108,NaN,NaN,768,...,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN


#### add perceived time info
- Settings for TAP run:
    - initial_wait_perception_factor = 1.5
    - transfer_wait_perception_factor = 3.0
    - walk_perception_factor = 2.0
    - in_vehicle_perception_factor = 1.0
    - initial_boarding_penalty = 10
    - transfer_boarding_penalty = 10

In [13]:
TAP_INITIAL_WAIT_PERCEPTION_FACTOR = 1.5
TAP_TRANSFER_WAIT_PERCEPTION_FACTOR = 3.0
TAP_WALK_PERCEPTION_FACTOR = 2.0
TAP_IN_VEHICLE_PERCEPTION_FACTOR = 1.0
TAP_INITIAL_BOARDING_PENALTY = 10
TAP_TRANSFER_BOARDING_PENALTY = 10

In [14]:
# add perception total time
tap_approach_db["TOTAL_PERCEIVED_TIME"] = TAP_INITIAL_WAIT_PERCEPTION_FACTOR * tap_approach_db["FIRSTWAIT"] + TAP_TRANSFER_WAIT_PERCEPTION_FACTOR * tap_approach_db["XFERWAIT"] + TAP_WALK_PERCEPTION_FACTOR * (tap_approach_db["access_time"] + tap_approach_db["egress_time"] + tap_approach_db["XFERWALK"]) + TAP_IN_VEHICLE_PERCEPTION_FACTOR * tap_approach_db["TOTALIVTT"] + TAP_INITIAL_BOARDING_PENALTY + TAP_TRANSFER_BOARDING_PENALTY * tap_approach_db["XFERS"]

In [15]:
## add maz coordinates
tap_approach_db = pd.merge(tap_approach_db, maz_nodes, how="left", left_on="maz_o", right_on="maz")
tap_approach_db = tap_approach_db.rename(columns={"lon": "maz_o_lon", "lat": "maz_o_lat"}).drop(columns="maz")
tap_approach_db = pd.merge(tap_approach_db, maz_nodes, how="left", left_on="maz_d", right_on="maz")
tap_approach_db = tap_approach_db.rename(columns={"lon": "maz_d_lon", "lat": "maz_d_lat"}).drop(columns="maz")
tap_approach_db = tap_approach_db[['unique_ID', 'maz_o', 'maz_d', 'maz_o_lon',
       'maz_o_lat', 'maz_d_lon', 'maz_d_lat', 'period', 'skim_set', 'tap_o', 'tap_d',
       'access_time', 'egress_time', 'renum_tap_o', 'renum_tap_d', 'CAPPEN',
       'CRIVTT', 'CROWD', 'EAWT', 'EBIVTT', 'FARE', 'FIRSTWAIT', 'FRIVTT',
       'HRIVTT', 'LBIVTT', 'LINKREL', 'LRIVTT', 'TOTALIVTT', 'TOTALWAIT',
       'TOTALWALK', 'XFERS', 'XFERWAIT', 'XFERWALK', 'TOTAL_TIME', 'TOTAL_PERCEIVED_TIME']]
tap_approach_db.head()

,unique_ID,maz_o,maz_d,maz_o_lon,maz_o_lat,maz_d_lon,maz_d_lat,period,skim_set,tap_o,...,LINKREL,LRIVTT,TOTALIVTT,TOTALWAIT,TOTALWALK,XFERS,XFERWAIT,XFERWALK,TOTAL_TIME,TOTAL_PERCEIVED_TIME
0,11420___AC Transit___2018,413852,414639,-122.299767,37.904587,-122.319763,37.979455,EA,1,490058,...,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,11423___AC Transit___2018,424701,414651,-122.326702,37.921951,-122.321789,37.986542,EA,1,490040,...,0.0,0.0,11.970170,30.0,10.805839,2.0,15.0,10.145839,60.636009,146.801847
2,11425___AC Transit___2018,418818,424101,-122.362677,37.937884,-122.328843,37.990794,EA,1,490074,...,0.0,0.0,10.733758,30.0,2.304314,1.0,15.0,1.644314,49.158072,115.082385
3,11730___AC Transit___2018,322305,329001,-122.111420,37.698752,-122.230930,37.798295,EA,1,390570,...,0.0,0.0,18.925690,15.0,0.400000,0.0,0.0,0.000000,68.245690,120.065690
4,11732___AC Transit___2018,310894,329001,-122.230286,37.788432,-122.230930,37.798295,EA,1,390659,...,0.0,0.0,2.697566,15.0,0.400000,0.0,0.0,0.000000,23.337566,46.477566


### Create skim db for TAZ approach
#### import TAZ lookups

In [17]:
cube_taz_lookup = pd.read_csv(f"{data_path}/maz_data_withDensity.csv")
cube_taz_lookup = cube_taz_lookup[["MAZ_ORIGINAL", "TAZ_ORIGINAL"]].rename(columns={"MAZ_ORIGINAL": "maz", "TAZ_ORIGINAL": "taz"})
cube_taz_lookup.head()

,maz,taz
0,10001,56
1,10002,56
2,10003,10
3,10004,53
4,10005,48


In [18]:
emme_taz_lookup = pd.read_csv(f"{data_path}/taz_approach/emme_taz_transit_network_node_id_crosswalk.csv").rename(columns={"emme_node_id": "emme_taz", "model_node_id": "cube_taz"})
emme_taz_lookup.head()

,emme_taz,cube_taz
0,1,1
1,2,2
2,3,3
3,4,4
4,5,5


#### import reformatted on-board survey record for the TAZ approach

In [19]:
taz_base = pd.read_csv(f"{data_path}/taz_approach/taz_approach_base_table.csv").drop(columns=["taz_o", "taz_d"])
taz_base = pd.merge(taz_base, cube_taz_lookup, how="left", left_on="maz_o", right_on="maz").drop(columns="maz").rename(columns={"taz": "cube_taz_o"})
taz_base = pd.merge(taz_base, cube_taz_lookup, how="left", left_on="maz_d", right_on="maz").drop(columns="maz").rename(columns={"taz": "cube_taz_d"})
taz_base = pd.merge(taz_base, emme_taz_lookup, how="left", left_on="cube_taz_o", right_on="cube_taz").drop(columns="cube_taz").rename(columns={"emme_taz": "emme_taz_o"})
taz_base = pd.merge(taz_base, emme_taz_lookup, how="left", left_on="cube_taz_d", right_on="cube_taz").drop(columns="cube_taz").rename(columns={"emme_taz": "emme_taz_d"})
taz_base = taz_base[["unique_ID", "emme_taz_o", "emme_taz_d", "maz_o", "maz_d", "period", "skim_set"]].rename(columns={"emme_taz_o": "taz_o", "emme_taz_d": "taz_d"})
taz_base.head()

,unique_ID,taz_o,taz_d,maz_o,maz_d,period,skim_set
0,1___AC Transit___2018,404,3620,11514,425106,PM,3
1,10___AC Transit___2018,2345,2878,316876,324992,AM,1
2,100___AC Transit___2018,2971,2138,318567,318470,PM,1
3,1000___AC Transit___2018,2312,2137,327175,325899,AM,1
4,10000___AC Transit___2018,2077,2481,323825,318288,MD,1


#### add estimated (1) walk access time for maz_o (2) walk egress time for maz_d

In [20]:
est_maz_walk_access_time = pd.read_csv(f"{data_path}/taz_approach/estimated_maz_access_walk_time.csv")
est_maz_walk_access_time = est_maz_walk_access_time.rename(columns={"time_period": "period", "from_maz": "maz_o", "est_maz_walk_access_min": "access_time"})
est_maz_walk_access_time.head()

,skim_set,period,maz_o,access_time
0,1,AM,10001,5.319
1,1,AM,10002,5.387
2,1,AM,10003,7.442
3,1,AM,10004,5.349
4,1,AM,10005,6.454


In [21]:
est_maz_walk_egress_time = pd.read_csv(f"{data_path}/taz_approach/estimated_maz_egress_walk_time.csv")
est_maz_walk_egress_time = est_maz_walk_egress_time.rename(columns={"time_period": "period", "to_maz": "maz_d", "est_maz_walk_egress_min": "egress_time"})
est_maz_walk_egress_time.head()

,skim_set,period,maz_d,egress_time
0,1,AM,10001,5.878
1,1,AM,10002,6.346
2,1,AM,10003,6.819
3,1,AM,10004,5.268
4,1,AM,10005,5.868


In [23]:
taz_base = pd.merge(taz_base, est_maz_walk_access_time, how="left", on=["period", "skim_set", "maz_o"])
taz_base = pd.merge(taz_base, est_maz_walk_egress_time, how="left", on=["period", "skim_set", "maz_d"])
taz_base.head()

,unique_ID,taz_o,taz_d,maz_o,maz_d,period,skim_set,access_time,egress_time
0,1___AC Transit___2018,404,3620,11514,425106,PM,3,7.719,5.306
1,10___AC Transit___2018,2345,2878,316876,324992,AM,1,6.181,3.450
2,100___AC Transit___2018,2971,2138,318567,318470,PM,1,2.991,7.930
3,1000___AC Transit___2018,2312,2137,327175,325899,AM,1,2.293,6.646
4,10000___AC Transit___2018,2077,2481,323825,318288,MD,1,9.902,5.191


### add corresponding taz-to-taz skim value from omx matrix

In [24]:
taz_skim_ea = omx.open_file(f"{data_path}/taz_approach/skims/transit_skims_ea.omx")
taz_skim_am = omx.open_file(f"{data_path}/taz_approach/skims/transit_skims_am.omx")
taz_skim_md = omx.open_file(f"{data_path}/taz_approach/skims/transit_skims_md.omx")
taz_skim_pm = omx.open_file(f"{data_path}/taz_approach/skims/transit_skims_pm.omx")
taz_skim_ev = omx.open_file(f"{data_path}/taz_approach/skims/transit_skims_ev.omx")

In [25]:
def get_taz_skim_value(period, skim_set, taz_o, taz_d):
    # initialize result dictionary
    table_names = ["CAPPEN", "CRIVTT", "CROWD", "EAWT", "EBIVTT", "FARE", "FIRSTWAIT", "FRIVTT", "HRIVTT", "LBIVTT", "LINKREL", "LRIVTT", "TOTALIVTT", "TOTALWAIT", "TOTALWALK", "XFERS", "XFERWAIT", "XFERWALK",]
    result = {}

    # select which omx skim file to use
    if period == "EA":
        per_skim = taz_skim_ea
    elif period == "AM":
        per_skim = taz_skim_am
    elif period == "MD":
        per_skim = taz_skim_md
    elif period == "PM":
        per_skim = taz_skim_pm
    elif period == "EV":
        per_skim = taz_skim_ev
    
    # set string for filtering appropriate skim table based on skim_set argument
    if skim_set == 1:
        skim_set_str = "BUS"
    elif skim_set == 2:
        skim_set_str = "PREM"
    elif skim_set == 3:
        skim_set_str = "ALLPEN"

    # populate result dictionary
    for table in table_names:
        result[table] = per_skim[f"{period.lower()}_{skim_set_str}_{table}"][taz_o - 1, taz_d - 1] # -1 because matrix starts from index 0
    
    return result

In [26]:
def add_taz_to_taz_skim_values(base_table):

    skim_value_list = []
    for i in range(len(base_table)):
        period = base_table.loc[i, "period"]
        skim_set = base_table.loc[i, "skim_set"]
        taz_o = base_table.loc[i, "taz_o"]
        taz_d = base_table.loc[i, "taz_d"]
        
        skim_value_list.append(get_taz_skim_value(period, skim_set, taz_o, taz_d))

    skim_values = pd.DataFrame(skim_value_list)

    return pd.concat([base_table, skim_values], axis = 1)

In [27]:
taz_approach_db = add_taz_to_taz_skim_values(taz_base)
print(len(taz_approach_db))
taz_approach_db.head()

97842


,unique_ID,taz_o,taz_d,maz_o,maz_d,period,skim_set,access_time,egress_time,CAPPEN,...,HRIVTT,LBIVTT,LINKREL,LRIVTT,TOTALIVTT,TOTALWAIT,TOTALWALK,XFERS,XFERWAIT,XFERWALK
0,1___AC Transit___2018,404,3620,11514,425106,PM,3,7.719,5.306,0.0,...,34.0,0.000000,0.0,0.0,34.000000,7.500000,1.743429,0.0,0.0,1.743429
1,10___AC Transit___2018,2345,2878,316876,324992,AM,1,6.181,3.450,0.0,...,0.0,11.932483,0.0,0.0,11.932483,3.586207,14.267323,0.0,0.0,14.267323
2,100___AC Transit___2018,2971,2138,318567,318470,PM,1,2.991,7.930,0.0,...,0.0,4.464317,0.0,0.0,4.464317,3.750000,34.519989,0.0,0.0,34.519989
3,1000___AC Transit___2018,2312,2137,327175,325899,AM,1,2.293,6.646,0.0,...,0.0,4.559168,0.0,0.0,4.559168,3.586207,0.000000,0.0,0.0,0.000000
4,10000___AC Transit___2018,2077,2481,323825,318288,MD,1,9.902,5.191,0.0,...,0.0,2.984965,0.0,0.0,2.984965,7.500000,33.521255,0.0,0.0,33.521255


In [28]:
# add total time column
taz_approach_db["TOTAL_TIME"] = taz_approach_db["access_time"] + taz_approach_db["egress_time"] + taz_approach_db["TOTALIVTT"] + taz_approach_db["TOTALWAIT"] + taz_approach_db["XFERWALK"]
taz_approach_db.head()

,unique_ID,taz_o,taz_d,maz_o,maz_d,period,skim_set,access_time,egress_time,CAPPEN,...,LBIVTT,LINKREL,LRIVTT,TOTALIVTT,TOTALWAIT,TOTALWALK,XFERS,XFERWAIT,XFERWALK,TOTAL_TIME
0,1___AC Transit___2018,404,3620,11514,425106,PM,3,7.719,5.306,0.0,...,0.000000,0.0,0.0,34.000000,7.500000,1.743429,0.0,0.0,1.743429,56.268429
1,10___AC Transit___2018,2345,2878,316876,324992,AM,1,6.181,3.450,0.0,...,11.932483,0.0,0.0,11.932483,3.586207,14.267323,0.0,0.0,14.267323,39.417012
2,100___AC Transit___2018,2971,2138,318567,318470,PM,1,2.991,7.930,0.0,...,4.464317,0.0,0.0,4.464317,3.750000,34.519989,0.0,0.0,34.519989,53.655306
3,1000___AC Transit___2018,2312,2137,327175,325899,AM,1,2.293,6.646,0.0,...,4.559168,0.0,0.0,4.559168,3.586207,0.000000,0.0,0.0,0.000000,17.084375
4,10000___AC Transit___2018,2077,2481,323825,318288,MD,1,9.902,5.191,0.0,...,2.984965,0.0,0.0,2.984965,7.500000,33.521255,0.0,0.0,33.521255,59.099220


In [29]:
# if TOTALIVTT == 0, set TOTAL_TIME = "NA"
# for walk-only paths, set all skim values to null
for column_name in ["access_time", "egress_time", "CAPPEN", "CRIVTT", "CROWD", "EAWT", "EBIVTT", "FARE", "FIRSTWAIT", "FRIVTT", "HRIVTT", "LBIVTT", "LINKREL", "LRIVTT", "TOTALWAIT", "TOTALWALK", "XFERS", "XFERWAIT", "XFERWALK", "TOTAL_TIME"]:
    taz_approach_db.loc[taz_approach_db["TOTALIVTT"] == 0, column_name] = np.nan
temp = taz_approach_db[taz_approach_db["TOTALIVTT"] == 0]
temp.head()

,unique_ID,taz_o,taz_d,maz_o,maz_d,period,skim_set,access_time,egress_time,CAPPEN,...,LBIVTT,LINKREL,LRIVTT,TOTALIVTT,TOTALWAIT,TOTALWALK,XFERS,XFERWAIT,XFERWALK,TOTAL_TIME
16,10011___AC Transit___2018,2876,2681,319133,312527,MD,1,NaN,NaN,NaN,...,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN
39,10033___AC Transit___2018,1243,2242,212251,314907,MD,1,NaN,NaN,NaN,...,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN
45,10039___AC Transit___2018,2212,2499,330901,313992,MD,1,NaN,NaN,NaN,...,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN
51,10044___AC Transit___2018,3571,3570,420964,415454,MD,1,NaN,NaN,NaN,...,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN
57,1005___AC Transit___2018,2363,2136,327261,315627,AM,1,NaN,NaN,NaN,...,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN


#### add perceived time info
- Settings for TAZ run:
    - initial_wait_perception_factor = 1.5
    - transfer_wait_perception_factor = 3.0
    - walk_perception_factor = 2.0
    - in_vehicle_perception_factor = 1.0
    - initial_boarding_penalty = 10
    - transfer_boarding_penalty = 40

In [30]:
TAZ_INITIAL_WAIT_PERCEPTION_FACTOR = 1.5
TAZ_TRANSFER_WAIT_PERCEPTION_FACTOR = 3.0
TAZ_WALK_PERCEPTION_FACTOR = 2.0
TAZ_IN_VEHICLE_PERCEPTION_FACTOR = 1.0
TAZ_INITIAL_BOARDING_PENALTY = 10
TAZ_TRANSFER_BOARDING_PENALTY = 40

In [31]:
# add perception total time
taz_approach_db["TOTAL_PERCEIVED_TIME"] = TAZ_INITIAL_WAIT_PERCEPTION_FACTOR * taz_approach_db["FIRSTWAIT"] + TAZ_TRANSFER_WAIT_PERCEPTION_FACTOR * taz_approach_db["XFERWAIT"] + TAZ_WALK_PERCEPTION_FACTOR * (taz_approach_db["access_time"] + taz_approach_db["egress_time"] + taz_approach_db["XFERWALK"]) + TAZ_IN_VEHICLE_PERCEPTION_FACTOR * taz_approach_db["TOTALIVTT"] + TAZ_INITIAL_BOARDING_PENALTY + TAZ_TRANSFER_BOARDING_PENALTY * taz_approach_db["XFERS"]

In [32]:
## add maz coordinates
taz_approach_db = pd.merge(taz_approach_db, maz_nodes, how="left", left_on="maz_o", right_on="maz")
taz_approach_db = taz_approach_db.rename(columns={"lon": "maz_o_lon", "lat": "maz_o_lat"}).drop(columns="maz")
taz_approach_db = pd.merge(taz_approach_db, maz_nodes, how="left", left_on="maz_d", right_on="maz")
taz_approach_db = taz_approach_db.rename(columns={"lon": "maz_d_lon", "lat": "maz_d_lat"}).drop(columns="maz")
taz_approach_db = taz_approach_db[['unique_ID', 'taz_o', 'taz_d', 'maz_o', 'maz_d', 'maz_o_lon',
       'maz_o_lat', 'maz_d_lon', 'maz_d_lat', 'period', 'skim_set',
       'access_time', 'egress_time', 'CAPPEN',
       'CRIVTT', 'CROWD', 'EAWT', 'EBIVTT', 'FARE', 'FIRSTWAIT', 'FRIVTT',
       'HRIVTT', 'LBIVTT', 'LINKREL', 'LRIVTT', 'TOTALIVTT', 'TOTALWAIT',
       'TOTALWALK', 'XFERS', 'XFERWAIT', 'XFERWALK', 'TOTAL_TIME', 'TOTAL_PERCEIVED_TIME']]
taz_approach_db.head()

,unique_ID,taz_o,taz_d,maz_o,maz_d,maz_o_lon,maz_o_lat,maz_d_lon,maz_d_lat,period,...,LINKREL,LRIVTT,TOTALIVTT,TOTALWAIT,TOTALWALK,XFERS,XFERWAIT,XFERWALK,TOTAL_TIME,TOTAL_PERCEIVED_TIME
0,1___AC Transit___2018,404,3620,11514,425106,-122.409795,37.773544,-122.345279,37.934689,PM,...,0.0,0.0,34.000000,7.500000,1.743429,0.0,0.0,1.743429,56.268429,84.786859
1,10___AC Transit___2018,2345,2878,316876,324992,-122.283143,37.814792,-122.199787,37.766251,AM,...,0.0,0.0,11.932483,3.586207,14.267323,0.0,0.0,14.267323,39.417012,75.108438
2,100___AC Transit___2018,2971,2138,318567,318470,-122.177856,37.714332,-122.165258,37.745239,PM,...,0.0,0.0,4.464317,3.750000,34.519989,0.0,0.0,34.519989,53.655306,110.971295
3,1000___AC Transit___2018,2312,2137,327175,325899,-122.197990,37.765282,-122.163331,37.737274,AM,...,0.0,0.0,4.559168,3.586207,0.000000,0.0,0.0,0.000000,17.084375,37.816478
4,10000___AC Transit___2018,2077,2481,323825,318288,-122.188702,37.793200,-122.247606,37.755097,MD,...,0.0,0.0,2.984965,7.500000,33.521255,0.0,0.0,33.521255,59.099220,121.463476


#### create `path_found` variable

In [34]:
tap_path_found = tap_approach_db.copy()
tap_path_found = tap_path_found[["unique_ID", "TOTALIVTT"]].rename(columns={"TOTALIVTT": "tap_TOTALIVTT"})
taz_path_found = taz_approach_db.copy()
taz_path_found = taz_path_found[["unique_ID", "TOTALIVTT"]].rename(columns={"TOTALIVTT": "taz_TOTALIVTT"})

In [35]:
path_found = pd.merge(taz_path_found, tap_path_found, how="left", on="unique_ID")
path_found["tap_TOTALIVTT"] = path_found["tap_TOTALIVTT"].fillna(0)
path_found.loc[(path_found["tap_TOTALIVTT"] == 0) & (path_found["taz_TOTALIVTT"] == 0), "path_found"] = "NEITHER"
path_found.loc[(path_found["tap_TOTALIVTT"] == 0) & (path_found["taz_TOTALIVTT"] > 0), "path_found"] = "TAZ_ONLY"
path_found.loc[(path_found["tap_TOTALIVTT"] > 0) & (path_found["taz_TOTALIVTT"] == 0), "path_found"] = "TAP_ONLY"
path_found.loc[(path_found["tap_TOTALIVTT"] > 0) & (path_found["taz_TOTALIVTT"] > 0), "path_found"] = "BOTH"
path_found = path_found[["unique_ID", "path_found"]]
path_found.head()

,unique_ID,path_found
0,1___AC Transit___2018,TAZ_ONLY
1,10___AC Transit___2018,BOTH
2,100___AC Transit___2018,BOTH
3,1000___AC Transit___2018,BOTH
4,10000___AC Transit___2018,BOTH


### Convert to long format
#### tap approach in long format

In [36]:
tap_approach_db_long = tap_approach_db.drop(columns=["tap_o", "tap_d", "renum_tap_o", "renum_tap_d"])
tap_approach_db_long = pd.melt(tap_approach_db_long, 
                               id_vars=["unique_ID", "maz_o", "maz_d", "maz_o_lon", "maz_o_lat", "maz_d_lon", "maz_d_lat", "period", "skim_set"], 
                               value_vars=["access_time", "egress_time", "CAPPEN", "CRIVTT", "CROWD", "EAWT", "EBIVTT", "FARE", "FIRSTWAIT", "FRIVTT", "HRIVTT", "LBIVTT", "LINKREL", "LRIVTT", "TOTALIVTT", "TOTALWAIT", "TOTALWALK", "XFERS", "XFERWAIT", "XFERWALK", "TOTAL_TIME", "TOTAL_PERCEIVED_TIME"],
                               ignore_index=False)
tap_approach_db_long = tap_approach_db_long.rename(columns={"variable": "skim_table", "value": "tap_value"})
tap_approach_db_long.head()

,unique_ID,maz_o,maz_d,maz_o_lon,maz_o_lat,maz_d_lon,maz_d_lat,period,skim_set,skim_table,tap_value
0,11420___AC Transit___2018,413852,414639,-122.299767,37.904587,-122.319763,37.979455,EA,1,access_time,NaN
1,11423___AC Transit___2018,424701,414651,-122.326702,37.921951,-122.321789,37.986542,EA,1,access_time,3.24
2,11425___AC Transit___2018,418818,424101,-122.362677,37.937884,-122.328843,37.990794,EA,1,access_time,5.52
3,11730___AC Transit___2018,322305,329001,-122.111420,37.698752,-122.230930,37.798295,EA,1,access_time,15.18
4,11732___AC Transit___2018,310894,329001,-122.230286,37.788432,-122.230930,37.798295,EA,1,access_time,3.12


#### taz approach in long format

In [37]:
taz_approach_db_long = taz_approach_db.drop(columns=["taz_o", "taz_d"])
taz_approach_db_long = pd.melt(taz_approach_db_long, 
                               id_vars=["unique_ID", "maz_o", "maz_d", "maz_o_lon", "maz_o_lat", "maz_d_lon", "maz_d_lat", "period", "skim_set"], 
                               value_vars=["access_time", "egress_time", "CAPPEN", "CRIVTT", "CROWD", "EAWT", "EBIVTT", "FARE", "FIRSTWAIT", "FRIVTT", "HRIVTT", "LBIVTT", "LINKREL", "LRIVTT", "TOTALIVTT", "TOTALWAIT", "TOTALWALK", "XFERS", "XFERWAIT", "XFERWALK", "TOTAL_TIME", "TOTAL_PERCEIVED_TIME"],
                               ignore_index=False)
taz_approach_db_long = taz_approach_db_long.rename(columns={"variable": "skim_table", "value": "taz_value"})
taz_approach_db_long.head()

,unique_ID,maz_o,maz_d,maz_o_lon,maz_o_lat,maz_d_lon,maz_d_lat,period,skim_set,skim_table,taz_value
0,1___AC Transit___2018,11514,425106,-122.409795,37.773544,-122.345279,37.934689,PM,3,access_time,7.719
1,10___AC Transit___2018,316876,324992,-122.283143,37.814792,-122.199787,37.766251,AM,1,access_time,6.181
2,100___AC Transit___2018,318567,318470,-122.177856,37.714332,-122.165258,37.745239,PM,1,access_time,2.991
3,1000___AC Transit___2018,327175,325899,-122.197990,37.765282,-122.163331,37.737274,AM,1,access_time,2.293
4,10000___AC Transit___2018,323825,318288,-122.188702,37.793200,-122.247606,37.755097,MD,1,access_time,9.902


### Combine results of both approaches into a single database

In [38]:
skim_comparison_db = pd.merge(taz_approach_db_long, tap_approach_db_long, how="outer", on=["unique_ID", "maz_o", "maz_d", "maz_o_lon", "maz_o_lat", "maz_d_lon", "maz_d_lat", "period", "skim_set", "skim_table"])
skim_comparison_db.head(10)

,unique_ID,maz_o,maz_d,maz_o_lon,maz_o_lat,maz_d_lon,maz_d_lat,period,skim_set,skim_table,taz_value,tap_value
0,1___AC Transit___2018,11514,425106,-122.409795,37.773544,-122.345279,37.934689,PM,3,access_time,7.719,NaN
1,10___AC Transit___2018,316876,324992,-122.283143,37.814792,-122.199787,37.766251,AM,1,access_time,6.181,22.38
2,100___AC Transit___2018,318567,318470,-122.177856,37.714332,-122.165258,37.745239,PM,1,access_time,2.991,2.88
3,1000___AC Transit___2018,327175,325899,-122.197990,37.765282,-122.163331,37.737274,AM,1,access_time,2.293,2.88
4,10000___AC Transit___2018,323825,318288,-122.188702,37.793200,-122.247606,37.755097,MD,1,access_time,9.902,9.84
5,10001___AC Transit___2018,312489,330362,-122.286719,37.778156,-122.075637,37.609550,MD,1,access_time,5.376,NaN
6,10002___AC Transit___2018,316488,314634,-122.204659,37.785926,-122.161125,37.791154,MD,1,access_time,7.014,7.86
7,10003___AC Transit___2018,314999,312527,-122.012347,37.588927,-121.898255,37.511935,MD,3,access_time,NaN,NaN
8,10004___AC Transit___2018,319567,322475,-122.218050,37.789919,-122.241402,37.767258,MD,1,access_time,5.210,4.14
9,10005___AC Transit___2018,312905,319173,-122.100097,37.632123,-122.059526,37.632255,MD,1,access_time,8.378,6.48


In [39]:
## add survey operator & route info
survey_info = pd.read_csv(f"{data_path}/survey_operator_route.csv", encoding = "ISO-8859-1")
skim_comparison_db = pd.merge(skim_comparison_db, survey_info, how="left", on="unique_ID")
skim_comparison_db = skim_comparison_db[["unique_ID", "operator", "route", "maz_o", "maz_d", "maz_o_lon", "maz_o_lat", "maz_d_lon", "maz_d_lat", "period", "skim_set", "skim_table", "tap_value", "taz_value"]]
skim_comparison_db.head()

,unique_ID,operator,route,maz_o,maz_d,maz_o_lon,maz_o_lat,maz_d_lon,maz_d_lat,period,skim_set,skim_table,tap_value,taz_value
0,1___AC Transit___2018,AC Transit,AC TRANSIT___72M Point Richmond to Oakland Amtrak,11514,425106,-122.409795,37.773544,-122.345279,37.934689,PM,3,access_time,NaN,7.719
1,10___AC Transit___2018,AC Transit,AC TRANSIT___1 Berkeley BART to Bay Fair BART,316876,324992,-122.283143,37.814792,-122.199787,37.766251,AM,1,access_time,22.38,6.181
2,100___AC Transit___2018,AC Transit,AC TRANSIT___1 Berkeley BART to Bay Fair BART,318567,318470,-122.177856,37.714332,-122.165258,37.745239,PM,1,access_time,2.88,2.991
3,1000___AC Transit___2018,AC Transit,AC TRANSIT___1 Berkeley BART to Bay Fair BART,327175,325899,-122.197990,37.765282,-122.163331,37.737274,AM,1,access_time,2.88,2.293
4,10000___AC Transit___2018,AC Transit,AC TRANSIT___19 Downtown Oakland Fruitvale BART,323825,318288,-122.188702,37.793200,-122.247606,37.755097,MD,1,access_time,9.84,9.902


In [40]:
## add path_found variable
skim_comparison_db = pd.merge(skim_comparison_db, path_found, how="left", on="unique_ID")
skim_comparison_db.head()

,unique_ID,operator,route,maz_o,maz_d,maz_o_lon,maz_o_lat,maz_d_lon,maz_d_lat,period,skim_set,skim_table,tap_value,taz_value,path_found
0,1___AC Transit___2018,AC Transit,AC TRANSIT___72M Point Richmond to Oakland Amtrak,11514,425106,-122.409795,37.773544,-122.345279,37.934689,PM,3,access_time,NaN,7.719,TAZ_ONLY
1,10___AC Transit___2018,AC Transit,AC TRANSIT___1 Berkeley BART to Bay Fair BART,316876,324992,-122.283143,37.814792,-122.199787,37.766251,AM,1,access_time,22.38,6.181,BOTH
2,100___AC Transit___2018,AC Transit,AC TRANSIT___1 Berkeley BART to Bay Fair BART,318567,318470,-122.177856,37.714332,-122.165258,37.745239,PM,1,access_time,2.88,2.991,BOTH
3,1000___AC Transit___2018,AC Transit,AC TRANSIT___1 Berkeley BART to Bay Fair BART,327175,325899,-122.197990,37.765282,-122.163331,37.737274,AM,1,access_time,2.88,2.293,BOTH
4,10000___AC Transit___2018,AC Transit,AC TRANSIT___19 Downtown Oakland Fruitvale BART,323825,318288,-122.188702,37.793200,-122.247606,37.755097,MD,1,access_time,9.84,9.902,BOTH


In [ ]:
skim_comparison_db.to_csv("../../outputs/skim/skim_comparison.csv", index=False)